# CAFA training sister analysis

Set WP1_ROOT and WP1_STAGE02_RUN_ID before starting the kernel. This workflow replays the documented census and pruning assignments from the existing FastOMA run. See the stage README for setup.

# CAFA-training HOG sisters and unary-retaining export

12 November 2026. Select, analyse and filter CAFA-training root HOGs in one notebook. The historical unary-collapse run and its outputs remain available for comparison. Run with pandas, matplotlib, lxml and OrthoXML-tools 1.3.1. Set HOGPROP_PYTHON to a Python interpreter with HOGprop’s dependencies if they are in a separate environment.

## 1. Identify the CAFA-training cohort

Stream the original FastOMA OrthoXML. Training proteins have numeric CAFA taxon labels and a train suffix. Select every whole root HOG that references at least one and record the exact XML IDs. Verify the input hash and reconcile the selected roots against the earlier census.

In [ ]:
from collections import Counter, defaultdict
import csv
import hashlib
import importlib.util
import json
import os
from pathlib import Path
import subprocess
import sys
from xml.parsers import expat

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
from lxml import etree

import re

ROOT = Path(os.environ["WP1_ROOT"]).expanduser()
if not ROOT.is_absolute():
    raise ValueError("WP1_ROOT must be an absolute workspace path")
REPO = ROOT / "HOGprop_validation_project"
STAGE = REPO / "analysis/02_fastoma_output_processing"
run_id = os.environ["WP1_STAGE02_RUN_ID"]
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", run_id):
    raise ValueError("Invalid WP1_STAGE02_RUN_ID")

HIST = STAGE / "data/reference"
OUT = STAGE / "results" / run_id
SISTER_OUT = OUT / "sister_analysis"
CENSUS = HIST / "all_hog_tree_statistics.tsv"
SOURCE = REPO / "analysis/01_fastoma/results/20261105_101624_7305523/FastOMA_HOGs.orthoxml"
XML = OUT / "cafa_train_unpruned.orthoxml"
MAP = REPO / "irene_cafa_oma_fasta_pairs.tsv"
PROFESSOR = STAGE / "scripts/orthoxml_subset.py"
for required in (SOURCE, MAP, PROFESSOR, CENSUS, HIST / "pruning/summary.json",
                 HIST / "pruning/pruning_assignments.tsv"):
    if not required.is_file():
        raise FileNotFoundError(required)
SISTER_OUT.parent.parent.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=False)
SISTER_OUT.mkdir()
SOURCE_COLS = ["OMA_leaves", "CAFA_train_leaves", "CAFA_test_leaves"]

def show_figure(fig, name, directory=OUT):
    path = directory / f"{name}.png"
    fig.savefig(path, dpi=145, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(path)))

NS = "http://orthoXML.org/2011/"
def q(name):
    return f"{{{NS}}}{name}"
ORTHO, PARA, REF, GENE, GROUPS = map(q, ("orthologGroup", "paralogGroup", "geneRef", "gene", "groups"))
GROUP_TAGS = {ORTHO, PARA}
TREE_TAGS = GROUP_TAGS | {REF}
END_TAGS = (*GROUP_TAGS, GROUPS, q("species"), q("taxonomy"), q("scores"), q("notes"))

def records(path):
    context = etree.iterparse(str(path), events=("start", "end"), tag=(q("orthoXML"), *END_TAGS),
                              remove_blank_text=True, resolve_entities=False, no_network=True)
    _, root = next(context)
    assert root.tag == q("orthoXML")
    yield "header", root
    for action, node in context:
        parent = node.getparent()
        if action == "start":
            if node.tag == GROUPS:
                yield "start_groups", node
        elif parent is root or (node.tag in GROUP_TAGS and parent is not None and parent.tag == GROUPS):
            tag = "end_groups" if node.tag == GROUPS else etree.QName(node).localname
            yield tag, node
            node.clear()
            while node.getprevious() is not None:
                del parent[0]

def file_hash(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def tree_hash(node):
    return hashlib.sha256(etree.tostring(node, with_tail=False)).digest()

source_sha = file_hash(SOURCE)
prior = json.loads((HIST / "pruning/summary.json").read_text())
assert source_sha == prior["xml_sha256"]
training_genes = {}
root_map = {}
original_tree_hashes = {}
source_roots = 0
stage = OUT / ".selected_groups.orthoxml"
with etree.xmlfile(str(stage), encoding="UTF-8") as writer:
    with writer.element(q("orthoXML"), nsmap={None: NS}):
        with writer.element(GROUPS):
            for kind, node in records(SOURCE):
                if kind == "species" and node.get("name").isdigit():
                    taxon = node.get("name")
                    for gene in node.iter(GENE):
                        protein = gene.get("protId")
                        _, mapped_taxon, partition = protein.rsplit("_", 2)
                        assert taxon == mapped_taxon and partition in {"train", "test"}
                        if partition == "train":
                            gid = gene.get("id")
                            assert gid not in training_genes
                            training_genes[gid] = (protein, taxon)
                elif kind in ("orthologGroup", "paralogGroup"):
                    source_roots += 1
                    exact = node.get("id")
                    if any(ref.get("id") in training_genes for ref in node.iter(REF)):
                        assert kind == "orthologGroup"
                        hog = exact.split("_", 1)[0]
                        assert hog not in root_map
                        root_map[hog] = exact
                        original_tree_hashes[exact] = tree_hash(node)
                        writer.write(node)

census = pd.read_csv(CENSUS, sep="\t", usecols=["root_hog", "protein_leaves", *SOURCE_COLS])
assert census.root_hog.is_unique and source_roots == len(census)
assert census[SOURCE_COLS].sum(axis=1).eq(census.protein_leaves).all()
train = census.loc[census.CAFA_train_leaves.gt(0)].copy().set_index("root_hog")
assert root_map.keys() == set(train.index)
exact_ids = set(root_map.values())
ids_file = OUT / "root_hog_ids.txt"
ids_file.write_text("".join(f"{exact}\n" for exact in root_map.values()))
pd.Series(root_map, name="orthoxml_id").rename_axis("root_hog").to_csv(
    OUT / "root_hog_id_map.tsv", sep="\t"
)
train.to_csv(OUT / "HOGtree_train.tsv", sep="\t")
display(pd.Series({"Input CAFA-training proteins": len(training_genes),
                   "CAFA-training proteins in selected HOGs": int(train.CAFA_train_leaves.sum()),
                   "Selected root HOGs": len(train)}).to_frame("Count"))

selection = pd.Series({"HOGtree train": len(train), "Other root HOGs": len(census)-len(train)})
display(train[SOURCE_COLS].sum().to_frame("Protein leaves in HOGtree train"))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, counts, title in [
    (axes[0], selection, "Root HOG selection"),
    (axes[1], train[SOURCE_COLS].sum().rename(index=dict(zip(SOURCE_COLS, ["OMA", "CAFA train", "CAFA test"]))), "Leaves in selected trees"),
]:
    bars = ax.bar(counts.index, counts.values, color="#41758a")
    ax.set_title(title)
    ax.bar_label(bars, labels=[f"{n:,}" for n in counts], padding=3)
    ax.margins(y=.22)
    ax.ticklabel_format(axis="y", style="plain")
fig.tight_layout()
show_figure(fig, "01_train_selection")

## 2. Extract the complete HOGs with the professor’s script

Run the professor’s reference discovery and group and species filters through the OrthoXML-tools writer. Stage the selected root groups for the reference lookup and use a streaming reader for the large source. Verify exact source tree fingerprints and that the subset registry equals the referenced genes.

In [ ]:
spec = importlib.util.spec_from_file_location("professor_subset", PROFESSOR)
professor = importlib.util.module_from_spec(spec)
spec.loader.exec_module(professor)

class StreamingProfessorFilter(professor.FilterHOGSubsetWithContainedGenes):
    def __enter__(self):
        self.records = records(self.source)
        _, root = next(self.records)
        self._ns = NS
        self.nsmap = {"": NS}
        self.root_tag = "orthoXML"
        self.root_attribs = dict(root.attrib)
        return self

    def __exit__(self, *exc):
        self.records.close()

    def parse(self):
        for tag, element in self.records:
            if tag == "species":
                element = self.process_species(element)
                for database in element.findall(q("database")):
                    if next(database.iter(GENE), None) is None:
                        element.remove(database)
                if element.find(q("database")) is None:
                    continue
            elif tag in ("orthologGroup", "paralogGroup"):
                element = self.process_toplevel_group(element)
                if element is None:
                    continue
            yield tag, None if tag in ("start_groups", "end_groups") else element

# The professor's first pass only needs the selected groups. Stage those groups
# without the large source gene registry, then run the original two filter hooks.
refs = professor.find_gene_refs_in_selection(str(stage), professor.get_hog_list(ids_file))
stage.unlink()
assert len(refs) == int(train.protein_leaves.sum())
pending = XML.with_suffix(".pending.orthoxml")
professor.process_stream_orthoxml(
    SOURCE, str(pending), parser_cls=StreamingProfessorFilter,
    parser_kwargs={"roothog_ids": exact_ids, "gene_ids": refs},
)

def audit_xml(path, hashes, excluded_ids=()):
    genes, referenced, roots = set(), set(), set()
    for tag, node in records(path):
        if tag == "species":
            ids = [gene.get("id") for gene in node.iter(GENE)]
            assert ids and len(set(ids)) == len(ids) and genes.isdisjoint(ids)
            genes.update(ids)
        elif tag == "orthologGroup":
            exact = node.get("id")
            assert exact not in roots and tree_hash(node) == hashes[exact], exact
            roots.add(exact)
            ids = [ref.get("id") for ref in node.iter(REF)]
            assert len(set(ids)) == len(ids) and referenced.isdisjoint(ids)
            referenced.update(ids)
        else:
            assert tag != "paralogGroup"
    assert roots == hashes.keys() and genes == referenced and genes.isdisjoint(excluded_ids)
    return len(genes)

assert audit_xml(pending, original_tree_hashes) == len(refs)
pending.replace(XML)
del refs
print(f"Professor's subset verified for {len(exact_ids):,} complete root HOGs.", flush=True)


## 3. Compare CAFA-training and OMA leaf counts

Dataset 1 contains trees with at most as many CAFA-training leaves as OMA leaves. Dataset 2 contains trees with fewer OMA leaves than CAFA-training leaves. Tied trees belong to Dataset 1, and their CAFA-training leaves are the focal leaves. Every selected tree has a training leaf, so any tree with zero OMA leaves falls in dataset 2. Those trees contribute zero minority leaves and have undefined per-tree fractions.

In [ ]:
D1, D2 = "Dataset 1", "Dataset 2"
delta = train.CAFA_train_leaves - train.OMA_leaves
train["dataset"] = np.where(delta.le(0), D1, D2)
train["minority_leaves"] = train[["OMA_leaves", "CAFA_train_leaves"]].min(axis=1)
colors = {D1: "#426f9b", D2: "#c27744"}
composition = train.groupby("dataset").agg(
    trees=("dataset", "size"), OMA_leaves=("OMA_leaves", "sum"),
    CAFA_train_leaves=("CAFA_train_leaves", "sum"),
    zero_minority_trees=("minority_leaves", lambda x: x.eq(0).sum())
).reindex([D1, D2])
display(composition)
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
bars = left.bar(composition.index, composition.trees, color=[colors[k] for k in composition.index])
left.bar_label(bars, labels=[f"{n:,}" for n in composition.trees], padding=3)
left.margins(y=.18)
left.set_ylabel("Root HOG trees")
left.set_title("Trees by relative abundance")
for label, rows in train.groupby("dataset"):
    right.scatter(rows.CAFA_train_leaves, rows.OMA_leaves, s=8, alpha=.30,
                  color=colors[label], label=label)
right.set_xscale("symlog", linthresh=1)
right.set_yscale("symlog", linthresh=1)
right.set_xlabel("CAFA-training leaves per tree")
right.set_ylabel("OMA leaves per tree")
right.legend()
right.set_title("Source counts in each tree")
fig.tight_layout()
show_figure(fig, "02_source_balance", SISTER_OUT)

## 4. Find same-taxon direct sisters

Stream the verified unpruned OrthoXML subset. A numeric species leaf supplies the CAFA taxon ID, and the protein ID suffix identifies training membership. The overlap table maps OMA genome codes to NCBI IDs. Only its local ID fields are used. The parser collects the immediate protein children of each XML parent. A child group remains an internal node and its descendant proteins cannot qualify as direct sisters.

For each parent, count CAFA-training and OMA direct leaves by mapped taxon. A focal leaf (CAFA training in Dataset 1 or OMA in Dataset 2) qualifies if at least one direct leaf from the other source shares that taxon. Count the leaf once even if several sisters qualify. The parent may have more than two children. Ordinary paralog groups supply duplication matches. Ortholog groups supply speciation matches.

The denominator is **all focal leaves**, including leaves with no confirmed same-taxon sister. OMA codes absent from the overlap mapping cannot supply confirmed matches and their minority-leaf count is reported separately. A leaf has one immediate parent, so the match categories are disjoint.

In [ ]:
mapping_rows = pd.read_csv(MAP, sep="\t", dtype=str, usecols=["ncbi_taxid", "oma_code"])
assert mapping_rows.oma_code.is_unique
oma_taxon = dict(zip(mapping_rows.oma_code, mapping_rows.ncbi_taxid))
expected = {
    row.Index: ((row.OMA_leaves, row.CAFA_train_leaves, row.CAFA_test_leaves), row.dataset)
    for row in train.itertuples()
}

def direct_matches(children, minority_source):
    # Each child is an immediate protein leaf (source, mapped taxon, XML gene ID).
    other = "OMA" if minority_source == "CAFA train" else "CAFA train"
    other_taxa = {taxon for source, taxon, _ in children if source == other and taxon is not None}
    return [gid for source, taxon, gid in children
            if source == minority_source and taxon is not None and taxon in other_taxa]

# A synthetic multifurcation checks same-taxon matching and one count per focal leaf.
assert direct_matches([
    ("CAFA train", "9606", 1), ("OMA", "9606", 2), ("OMA", "9606", 3),
    ("CAFA train", "10090", 4), ("CAFA test", "9606", 5)
], "CAFA train") == [1]
assert direct_matches([("OMA", None, 1), ("CAFA train", "9606", 2)], "OMA") == []

class SisterCensus:
    def __init__(self):
        self.registry = {}
        self.train_ids = set()
        self.stack = []
        self.depth = 0
        self.root_count = 0
        self.seen = set()
        self.results = {}
        self.examples = {}
        self.registered = 0

    def start(self, tag, attrs):
        if tag == "species":
            self.species = attrs["name"]
            self.first = None
            self.n_genes = 0
        elif tag == "gene":
            gid = int(attrs["id"])
            if self.first is None:
                self.first = gid
            self.last = gid
            self.n_genes += 1
            self.registered += 1
            if self.species.isdigit():
                _, taxon, split = attrs["protId"].rsplit("_", 2)
                assert taxon == self.species and split in {"train", "test"}
                if split == "train":
                    self.train_ids.add(gid)
        elif tag in {"orthologGroup", "paralogGroup"}:
            if self.depth == 0:
                self.root_count += 1
                self.hog = attrs.get("id", "").split("_")[0]
                self.selected = self.hog in expected
                if self.selected:
                    self.counts = [0, 0, 0]
                    self.matches = Counter()
                    self.unknown_oma = 0
                    self.dataset = expected[self.hog][1]
                    self.minority = {D1: "CAFA train", D2: "OMA"}.get(self.dataset)
                    self.node_number = 0
            self.depth += 1
            if self.selected:
                if self.stack:
                    self.stack[-1]["degree"] += 1
                self.node_number += 1
                self.stack.append({"tag": tag, "kind": "", "direct": [],
                                   "degree": 0, "node": self.node_number})
        elif self.stack and tag == "property" and attrs.get("name") == "Type":
            self.stack[-1]["kind"] = attrs["value"]
        elif self.stack and tag == "geneRef":
            gid = int(attrs["id"])
            species, first, last = self.registry[gid // 1_000_000]
            assert first <= gid <= last
            if species.isdigit():
                category = 1 if gid in self.train_ids else 2
                source = "CAFA train" if category == 1 else "CAFA test"
                taxon = species
            else:
                category, source, taxon = 0, "OMA", oma_taxon.get(species)
                self.unknown_oma += taxon is None
            self.counts[category] += 1
            self.stack[-1]["degree"] += 1
            self.stack[-1]["direct"].append((source, taxon, gid))

    def end(self, tag):
        if tag == "species":
            last = self.last
            block = self.first // 1_000_000
            assert last // 1_000_000 == block and block not in self.registry
            self.registry[block] = (self.species, self.first, last)
        elif tag in {"orthologGroup", "paralogGroup"}:
            self.depth -= 1
            if self.selected:
                parent = self.stack.pop()
                event = ("speciation" if parent["tag"] == "orthologGroup"
                         else None if parent["kind"] else "duplication")
                if event is not None and parent["degree"] >= 2:
                    hits = direct_matches(parent["direct"], self.minority)
                    self.matches[event] += len(hits)
                    key = (self.dataset, event)
                    if hits and key not in self.examples and parent["degree"] <= 6:
                        self.examples[key] = (self.hog, parent.copy(), hits)
                if self.depth == 0:
                    assert tuple(self.counts) == expected[self.hog][0]
                    assert self.hog not in self.seen
                    self.seen.add(self.hog)
                    if self.minority is not None:
                        self.results[self.hog] = {
                            **{f"{event}_matched_leaves": self.matches[event]
                               for event in ("duplication", "speciation")},
                            "minority_OMA_unmapped": self.unknown_oma if self.minority == "OMA" else 0,
                        }

scan = SisterCensus()
parser = expat.ParserCreate()
parser.StartElementHandler = scan.start
parser.EndElementHandler = scan.end
xml_hash = hashlib.sha256()
with XML.open("rb") as handle:
    for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
        xml_hash.update(chunk)
        parser.Parse(chunk, False)
parser.Parse(b"", True)
assert scan.depth == 0 and scan.root_count == len(train)
assert scan.seen == set(train.index)
previous = json.loads((CENSUS.parent / "summary.json").read_text())
assert source_sha == previous["xml_sha256"]
print(f"Checked {scan.root_count:,} root HOGs, {len(scan.seen):,} selected trees, "
      f"and {scan.registered:,} registered protein records.")
print(f"All selected leaf counts match the earlier census. Mapping includes "
      f"{len(oma_taxon):,} OMA codes at {len(set(oma_taxon.values())):,} shared taxon IDs.")

## 5. Report both result datasets

Join the sister counts to the source census. Each row is one root HOG. The pooled fraction weights every focal leaf equally and uses the sum of matches divided by the sum of focal leaves. Per-tree distributions show how that fraction varies between trees. Zero-denominator trees are retained in the TSV and omitted from fraction histograms. A missing fraction is written as a blank field.

The matched counts under duplication and speciation parents are separate. The remainder includes leaves with no confirmed match under either parent type.

In [ ]:
matches = pd.DataFrame.from_dict(scan.results, orient="index")
matches.index.name = "root_hog"
result = train.join(matches, validate="one_to_one")
events = ["duplication", "speciation"]
match_cols = [f"{event}_matched_leaves" for event in events]
assert not result[match_cols].isna().any().any()
assert result[match_cols].sum(axis=1).le(result.minority_leaves).all()
result["no_confirmed_match_leaves"] = result.minority_leaves - result[match_cols].sum(axis=1)
denominator = result.minority_leaves.replace(0, np.nan)
for event in events:
    result[f"{event}_fraction"] = result[f"{event}_matched_leaves"] / denominator

summary_rows = []
for label, filename, minority in [
    (D1, "dataset1_CAFA_train_le_OMA.tsv", "CAFA train"),
    (D2, "dataset2_OMA_lt_CAFA_train.tsv", "OMA"),
]:
    rows = result.loc[result.dataset.eq(label)]
    rows.to_csv(SISTER_OUT / filename, sep="\t")
    pool = int(rows.minority_leaves.sum())
    summary_rows.append({
        "dataset": label, "minority_source": minority, "trees": len(rows),
        "trees_with_zero_minority": int(rows.minority_leaves.eq(0).sum()),
        "minority_leaves": pool, "unmapped_minority_OMA_leaves": int(rows.minority_OMA_unmapped.sum()),
        **{f"{ev}_matched_leaves": int(rows[f"{ev}_matched_leaves"].sum()) for ev in events},
        **{f"{ev}_fraction": float(rows[f"{ev}_matched_leaves"].sum() / pool)
           if pool else None for ev in events},
        "no_confirmed_match_leaves": int(rows.no_confirmed_match_leaves.sum()),
    })
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
    outcome = rows[match_cols + ["no_confirmed_match_leaves"]].sum()
    fractions = 100 * outcome / pool if pool else outcome * np.nan
    bars = left.bar(["Duplication", "Speciation", "No match"], fractions,
                   color=["#a75653", "#407f72", "#a2a8ae"])
    left.bar_label(bars, labels=[f"{v:,}\n({p:.2f}%)" for v, p in zip(outcome, fractions)], padding=3, fontsize=8)
    left.set_ylim(0, 122)
    left.set_ylabel("Percent of focal leaves")
    left.set_title(f"{label} · pooled outcomes")
    for event, color in [("duplication", "#a75653"), ("speciation", "#407f72")]:
        values = rows[f"{event}_fraction"].dropna()
        right.hist(values, bins=np.linspace(0, 1, 21), histtype="step", linewidth=2,
                   color=color, label=event.capitalize())
    right.set_yscale("log")
    right.set_xlabel("Fraction of focal leaves matched per tree")
    right.set_ylabel("Trees per bin (log scale)")
    right.set_title("Distribution across eligible trees")
    right.legend()
    fig.suptitle(f"Focal source: {minority} · {len(rows):,} trees · {pool:,} focal leaves")
    fig.tight_layout()
    show_figure(fig, f"0{3 if label == D1 else 4}_{filename[:-4]}", SISTER_OUT)
    display(rows[["OMA_leaves", "CAFA_train_leaves", "minority_leaves", *match_cols]].head())

summary = pd.DataFrame(summary_rows)
summary.to_csv(SISTER_OUT / "pooled_sister_summary.tsv", sep="\t", index=False)
display(summary)
assert len(result) == len(train)
assert result.dataset.eq(D1).eq(delta.le(0)).all()

### Ten duplication examples from cell 11

13 November 2026. The 63 observations count matched CAFA-training **leaves**, not necessarily distinct parents. Show the first ten distinct matching duplication parents, sorted by root HOG ID and preorder node number. Depth two means two child edges below the matching duplication (depth zero); deeper groups are collapsed and labelled with their full descendant protein count. All children within the displayed depth are retained. Red starred proteins are counted focal leaves; green proteins are their same-taxon direct OMA sisters. These are schematic, encoded HOG hierarchies, not measured evolutionary branch lengths.

This cell can run independently using the existing local unpruned subset and saved cell 11 table. It reconciles every positive per-tree count before plotting, leaves inputs unchanged, and writes separate PNGs and provenance in `unary_retained_20261112/duplication_examples_depth2_20261113/`. No extraction, pruning, export or cluster execution is needed.


In [ ]:
# Standalone local plotting cell: reads saved, unpruned results; no pipeline rerun.
import copy
import csv
import hashlib
import json
import platform
from pathlib import Path
import xml.etree.ElementTree as ET
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image

example_base = OUT
example_xml = XML
example_table = SISTER_OUT / 'dataset1_CAFA_train_le_OMA.tsv'
example_mapping = MAP
example_out = example_base / 'duplication_examples_depth2_20261113'
example_out.mkdir(exist_ok=True)
with example_table.open() as handle:
    example_rows = list(csv.DictReader(handle, delimiter='\t'))
example_expected = {r['root_hog']: int(r['duplication_matched_leaves']) for r in example_rows
                    if int(r['duplication_matched_leaves']) > 0}
with example_mapping.open() as handle:
    example_taxa = {r['oma_code']: r['ncbi_taxid'] for r in csv.DictReader(handle, delimiter='\t')}
example_ns = '{http://orthoXML.org/2011/}'
example_orth, example_para, example_ref = [example_ns + t for t in ('orthologGroup', 'paralogGroup', 'geneRef')]
example_groups = {example_orth, example_para}
example_tags = example_groups | {example_ref}
example_species, example_training = {}, set()
example_nodes, example_counts = [], {}
example_stack = []
# Keep only one species or root group in memory at a time.
for action, element in ET.iterparse(example_xml, events=('start', 'end')):
    if action == 'start':
        example_stack.append(element)
        continue
    parent = example_stack[-2] if len(example_stack) > 1 else None
    if element.tag == example_ns + 'species':
        sp = element.get('name')
        for gene in element.iter(example_ns + 'gene'):
            gid = int(gene.get('id'))
            example_species[gid // 1_000_000] = sp
            if sp.isdigit() and gene.get('protId').endswith('_train'):
                example_training.add(gid)
        parent.remove(element)
    elif element.tag in example_groups and parent.tag == example_ns + 'groups':
        hog = element.get('id').split('_', 1)[0]
        if hog in example_expected:
            matched = 0
            for number, node in enumerate((n for n in element.iter() if n.tag in example_groups), 1):
                if node.tag != example_para or any(
                    p.get('name') == 'Type' and p.get('value')
                    for p in node.findall(example_ns + 'property')
                ):
                    continue
                direct = [int(c.get('id')) for c in node if c.tag == example_ref]
                oma = {}
                for gid in direct:
                    sp = example_species[gid // 1_000_000]
                    if not sp.isdigit() and sp in example_taxa:
                        oma.setdefault(example_taxa[sp], []).append(gid)
                hits = [gid for gid in direct if gid in example_training
                        and example_species[gid // 1_000_000] in oma]
                matched += len(hits)
                if hits:
                    example_nodes.append((hog, number, copy.deepcopy(node), hits,
                        sorted({g for hit in hits for g in oma[example_species[hit // 1_000_000]]})))
            example_counts[hog] = matched
        parent.remove(element)
    example_stack.pop()
assert example_counts == example_expected, 'Mismatch with cell 11 per-tree duplication counts'
assert sum(example_counts.values()) == 63, 'The source cohort changed; review the example selection'
# Deterministic illustrative selection; distinct parents, not a random frequency sample.
example_nodes.sort(key=lambda item: (item[0], item[1]))
example_selected = example_nodes[:10]
assert len(example_selected) == 10
example_wanted = {int(ref.get('id')) for _, _, n, _, _ in example_selected for ref in n.iter(example_ref)}
example_names = {}
for action, element in ET.iterparse(example_xml, events=('start', 'end')):
    if action == 'start' and element.tag == example_ns + 'groups':
        break
    if action == 'end' and element.tag == example_ns + 'gene':
        gid = int(element.get('id'))
        if gid in example_wanted:
            example_names[gid] = element.get('protId')
    if action == 'end':
        element.clear()
assert example_names.keys() == example_wanted

def example_kind(node):
    if node.tag == example_orth:
        return 'Speciation-like'
    if any(p.get('name') == 'Type' and p.get('value') for p in node.findall(example_ns + 'property')):
        return 'Fragment group'
    return 'Duplication'

def example_draw(ax, root, hits, sisters):
    positions, drawn, edges = {}, [], []
    def layout(node, depth=0):
        children = [c for c in node if c.tag in example_tags] if depth < 2 else []
        ys = [layout(c, depth + 1) for c in children]
        y = sum(ys) / len(ys) if ys else len(drawn)
        if not ys:
            drawn.append(node)
        positions[node] = (depth, y)
        edges.extend((node, c) for c in children)
        return y
    layout(root)
    for a, b in edges:
        x1,y1 = positions[a]; x2,y2 = positions[b]
        ax.plot([x1,x1,x2], [y1,y2,y2], color='#a9b3bb', lw=1.2, zorder=1)
    for node, (depth,y) in positions.items():
        if node.tag == example_ref:
            gid = int(node.get('id')); sp = example_species[gid // 1_000_000]
            source = 'CAFA train' if gid in example_training else 'CAFA test' if sp.isdigit() else 'OMA'
            taxon = sp if sp.isdigit() else example_taxa.get(sp, 'unmapped')
            color = '#ae443d' if gid in hits else '#267d73' if gid in sisters else '#596875'
            label = f"{'* ' if gid in hits else ''}{example_names[gid]} | {source} | taxon {taxon}"
            ax.scatter(depth, y, s=35, color=color, zorder=3)
            ax.text(depth+.055, y, label, va='center', fontsize=8, color=color,
                    bbox=dict(facecolor='white', edgecolor='none', pad=1))
        else:
            collapsed = depth == 2
            label = example_kind(node)
            if collapsed:
                label += f" [collapsed: {sum(1 for _ in node.iter(example_ref)):,} proteins]"
            ax.scatter(depth, y, marker='>' if collapsed else 's', s=45,
                       color='#ae443d' if node.tag == example_para else '#526e85', zorder=3)
            ax.text(depth+.055, y-.18 if not collapsed else y, label,
                    va='bottom' if not collapsed else 'center', fontsize=8,
                    bbox=dict(facecolor='white', edgecolor='none', pad=1))
    ax.set_xlim(-.12, 3.9)
    ax.set_ylim(max(len(drawn),2)-.4, -.8)
    ax.axis('off')
    return len(drawn)

example_manifest = []
for i, (hog, number, node, hits, sisters) in enumerate(example_selected, 1):
    visible_tips = sum(max(1, sum(c.tag in example_tags for c in child))
                       if child.tag in example_groups else 1 for child in node if child.tag in example_tags)
    fig, ax = plt.subplots(figsize=(15, max(3.2, visible_tips*.42+1.6)))
    example_draw(ax, node, set(hits), set(sisters))
    fig.suptitle(f'Example {i}/10 | {hog} | duplication parent #{number}', fontsize=13, x=.04, ha='left')
    ax.set_title('Depth 2 from the duplication | red * = counted CAFA train; green = same-taxon OMA sister\n'
                 'Squares = groups; triangles = collapsed deeper branches; edge lengths are schematic',
                 fontsize=9, loc='left', pad=17)
    fig.tight_layout(rect=(0,0,1,.92))
    path = example_out / f'{i:02d}_{hog}_node{number}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(path)))
    example_manifest.append(dict(example=i, root_hog=hog, parent_node=number,
        cafa_gene_ids=hits, oma_sister_gene_ids=sisters, plot=path.name))

def example_sha(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

example_evidence = dict(date='2026-11-13', depth=2, depth_origin='matching duplication parent',
    selection='first 10 distinct parents sorted by root HOG ID then preorder node number',
    matched_focal_leaves=sum(example_counts.values()), distinct_matching_parents=len(example_nodes),
    input=str(example_xml), input_sha256=example_sha(example_xml),
    table_sha256=example_sha(example_table), mapping_sha256=example_sha(example_mapping),
    python=platform.python_version(), matplotlib=matplotlib.__version__, examples=example_manifest,
    validation='Every positive per-tree count reconciles exactly with the saved cell 11 table',
    limitation='Illustrative encoded hierarchy nodes; not independently validated biological duplications')
(example_out / 'evidence.json').write_text(json.dumps(example_evidence, indent=2)+'\n')
print(f"Verified {sum(example_counts.values())} matched CAFA-training leaves at {len(example_nodes)} distinct duplication parents; plotted 10.")


## 6. Inspect individual matches and verify the outputs

The following examples are the first readable matched parents encountered in the XML scan, with at most six immediate children. They illustrate the counting rule without representing frequencies. Each row shows an actual parent and its immediate protein leaves. The diagram highlights counted focal leaves with a star. Any additional child groups are shown as internal daughters. All reported frequencies above use the full census.

In [ ]:
examples = sorted(scan.examples.items())
if examples:
    fig, axes = plt.subplots(len(examples), 1, figsize=(11, 2.25 * len(examples)), squeeze=False)
    for ax, ((dataset, event), (hog, parent, hits)) in zip(axes[:, 0], examples):
        children = parent["direct"]
        nested = parent["degree"] - len(children)
        labels = [f"{'* ' if gid in hits else ''}{source}\nTaxon {taxon or 'unmapped'}\nGene ID {gid}"
                  for source, taxon, gid in children] + ["Child group"] * nested
        xs = np.linspace(.09, .91, len(labels))
        for x, label in zip(xs, labels):
            ax.plot([.5, x], [.72, .30], color="#667887")
            ax.text(x, .25, label, ha="center", va="top", fontsize=8,
                    color="#922c36" if label.startswith("*") else "#263746")
        ax.text(.5, .77, f"{dataset} · {hog} · {event} parent {parent['node']}", ha="center")
        ax.set(xlim=(0, 1), ylim=(-.16, 1))
        ax.axis("off")
    fig.tight_layout()
    show_figure(fig, "05_direct_sister_examples", SISTER_OUT)

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

evidence = {
    "date": "2026-11-12",
    "input_paths": {key: str(path.relative_to(ROOT)) for key, path in
                    [("orthoxml", SOURCE), ("subset_orthoxml", XML), ("census", CENSUS), ("mapping", MAP)]},
    "input_sha256": {"orthoxml": source_sha, "subset_orthoxml": xml_hash.hexdigest(), "census": sha256(CENSUS), "mapping": sha256(MAP)},
    "python": sys.version, "pandas": pd.__version__, "numpy": np.__version__,
    "matplotlib": matplotlib.__version__, "root_HOGs": scan.root_count,
    "HOGtree_train": len(train), "tied_trees_in_dataset_1": int(delta.eq(0).sum()), "datasets": summary_rows,
    "checks": ["source census sums match total protein leaves",
               "all subset leaf counts and root IDs match the source census",
               "OrthoXML hash matches prior validated export",
               "matched categories do not exceed minority-leaf counts",
               "synthetic direct-sister and unknown-mapping checks pass",
               "the two datasets partition the selected trees and all ties are in Dataset 1"],
}
(SISTER_OUT / "evidence.json").write_text(json.dumps(evidence, indent=2, allow_nan=False) + "\n")
display(pd.DataFrame({"Check passed": evidence["checks"]}))

### Interpretation

These fractions describe same-taxon source relationships in the exported HOG hierarchy. A direct sister under an encoded duplication or speciation parent does not independently establish a biological event. The taxonomy represents OMA and CAFA source collections separately, including when they share an NCBI taxon. This is a descriptive filtering exercise for the training cohort. Further GO masking and benchmark eligibility decisions belong to step 6.

## 7. Count OMA sister candidates at multifurcations

Use all CAFA-training protein leaves in the selected root HOGs, including trees where CAFA is the larger source and trees with tied source counts. A multifurcation has at least three immediate children, including any child groups. For each CAFA-training leaf at such a parent, count distinct OMA protein leaves that share the parent and mapped NCBI taxon ID. Distinct protein IDs identify distinct candidates. This check does not establish whether they represent the same gene.

The figure shows the numbers of focal CAFA-training leaves with zero, one, or multiple eligible OMA sisters. The table records the root HOG, parent, encoded event, taxon, and candidate XML gene IDs for review before pruning.

In [ ]:
def multifurcation_candidates(parent):
    if parent["degree"] < 3:
        return
    oma_by_taxon = defaultdict(set)
    for source, taxon, gene_id in parent["direct"]:
        if source == "OMA" and taxon is not None:
            oma_by_taxon[taxon].add(gene_id)
    for source, taxon, gene_id in parent["direct"]:
        if source == "CAFA train":
            yield gene_id, taxon, sorted(oma_by_taxon.get(taxon, ()))


example = {
    "degree": 6,
    "direct": [
        ("CAFA train", "9606", 1), ("OMA", "9606", 2), ("OMA", "9606", 3),
        ("CAFA train", "10090", 4), ("OMA", None, 5),
    ],
}
assert list(multifurcation_candidates(example)) == [
    (1, "9606", [2, 3]), (4, "10090", []),
]
assert list(multifurcation_candidates({"degree": 2, "direct": example["direct"][:2]})) == []


class MultifurcationCensus(SisterCensus):
    def __init__(self):
        super().__init__()
        self.candidates = []

    def end(self, tag):
        if self.stack and tag in {"orthologGroup", "paralogGroup"}:
            parent = self.stack[-1]
            if tag == "orthologGroup":
                event = "speciation"
            else:
                event = "fragment" if parent["kind"] else "duplication"
            for focal, taxon, sisters in multifurcation_candidates(parent):
                self.candidates.append({
                    "root_hog": self.hog,
                    "parent_node": parent["node"],
                    "event": event,
                    "parent_degree": parent["degree"],
                    "taxon_id": taxon,
                    "cafa_gene_id": focal,
                    "oma_sister_count": len(sisters),
                    "oma_gene_ids": ",".join(map(str, sisters)),
                })
        super().end(tag)


multifurcations = MultifurcationCensus()
parser = expat.ParserCreate()
parser.StartElementHandler = multifurcations.start
parser.EndElementHandler = multifurcations.end
multifurcation_hash = hashlib.sha256()
with XML.open("rb") as handle:
    for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
        multifurcation_hash.update(chunk)
        parser.Parse(chunk, False)
parser.Parse(b"", True)
assert multifurcations.depth == 0
assert multifurcations.root_count == len(train)
assert multifurcations.seen == set(train.index)
assert multifurcation_hash.hexdigest() == xml_hash.hexdigest()

columns = [
    "root_hog", "parent_node", "event", "parent_degree", "taxon_id",
    "cafa_gene_id", "oma_sister_count", "oma_gene_ids",
]
candidates = pd.DataFrame(multifurcations.candidates, columns=columns)
assert not candidates.duplicated(["root_hog", "cafa_gene_id"]).any()
assert len(candidates) <= train.CAFA_train_leaves.sum()
assert candidates.oma_gene_ids.map(
    lambda ids: len(ids.split(",")) if ids else 0
).eq(candidates.oma_sister_count).all()
candidates.to_csv(OUT / "multifurcation_oma_sister_candidates.tsv", sep="\t", index=False)

counts = candidates.oma_sister_count.clip(upper=2).value_counts().reindex(range(3), fill_value=0)
counts.index = ["Zero", "One", "Two or more"]
display(counts.to_frame("CAFA-training leaves at multifurcations"))
display(candidates.loc[candidates.oma_sister_count.gt(1)].head(10))
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(counts.index, counts.values, color="#41758a")
ax.bar_label(bars, labels=[f"{count:,}" for count in counts], padding=4)
ax.margins(y=.17)
ax.set_xlabel("Same-taxon direct OMA sisters")
ax.set_ylabel("CAFA-training leaves")
fig.tight_layout()
show_figure(fig, "06_oma_sisters_at_multifurcations")

## 8. Apply the saved OMA sister removals and retain unary nodes

Check each saved selection against its original parent, taxon and candidate IDs. Apply the same choices across both datasets. Remove each selected OMA gene reference once and preserve all groups, their labels and their parent relationships. Drop membership-dependent scores on groups affected by a deletion. Count whole-root singletons after removal and record their proteins as uncovered separately from unassigned CAFA-training proteins. One-protein subtrees within larger HOGs remain in the analysis.

In [ ]:
ASSIGNMENTS = HIST / "pruning/pruning_assignments.tsv"
FINAL = OUT / "cafa_train_unary_retained_no_singletons.orthoxml"
with ASSIGNMENTS.open() as handle:
    assignments = defaultdict(list)
    for row in csv.DictReader(handle, delimiter="\t"):
        if row["selected_oma_gene_id"]:
            assert row["event"] == "speciation"
            assignments[row["root_hog"]].append(row)
assert assignments.keys() <= root_map.keys()

def genome(gid):
    return scan.registry[int(gid) // 1_000_000][0]

def event(node):
    if node.tag == ORTHO:
        return "speciation"
    return ("fragment" if any(p.get("name") == "Type" for p in node.findall(q("property")))
            else "duplication")

def metrics_for(node):
    ids = [ref.get("id") for ref in node.iter(REF)]
    species = set(map(genome, ids))
    mapped = {oma_taxon.get(sp, sp) for sp in species}
    unmapped = sum(not sp.isdigit() and sp not in oma_taxon for sp in species)
    result = dict(leaves=len(ids), species_units=len(mapped), unmapped_oma_codes=unmapped,
                  duplications=0, multifurcations=0, speciations=0, unary_nodes=0)
    for group in node.iter(*GROUP_TAGS):
        degree = sum(child.tag in TREE_TAGS for child in group)
        assert degree > 0
        kind = event(group)
        result["duplications"] += kind == "duplication"
        result["speciations"] += kind == "speciation" and degree >= 2
        result["multifurcations"] += degree >= 3
        result["unary_nodes"] += degree == 1
    return ids, result

def apply_removals(node, hog, verify):
    if hog not in assignments:
        return set()
    lookup = {ref.get("id"): ref for ref in node.iter(REF)}
    if verify:
        numbers = {group: i for i, group in enumerate(node.iter(*GROUP_TAGS), 1)}
        for row in assignments[hog]:
            focal_id = row["cafa_gene_id"]
            target = row["selected_oma_gene_id"]
            focal, sister = lookup[focal_id], lookup[target]
            parent = focal.getparent()
            taxon = row["taxon_id"]
            assert focal_id in training_genes and training_genes[focal_id][1] == taxon
            assert parent is sister.getparent() and parent.tag == ORTHO
            assert numbers[parent] == int(row["parent_node"])
            candidates = {
                child.get("id") for child in parent if child.tag == REF
                and oma_taxon.get(genome(child.get("id"))) == taxon
            }
            assert candidates == set(row["candidate_oma_gene_ids"].split(",")) and target in candidates
    deleted = {row["selected_oma_gene_id"] for row in assignments[hog]}
    changed = set()
    for gid in deleted:
        ref = lookup[gid]
        parent = ref.getparent()
        changed.update(ref.iterancestors(*GROUP_TAGS))
        parent.remove(ref)
    for group in changed:
        for score in group.findall(q("score")):
            group.remove(score)
    return deleted

rows, singleton_rows = [], []
edited_hashes = {}
discard = set()
removed = set()
retained_training = set()
for tag, node in records(XML):
    if tag != "orthologGroup":
        continue
    exact = node.get("id")
    hog = exact.split("_", 1)[0]
    group_layout = [(g.tag, dict(g.attrib), g.getparent()) for g in node.iter(*GROUP_TAGS)]
    before_ids, before = metrics_for(node)
    deleted = apply_removals(node, hog, verify=True)
    after_ids, after = metrics_for(node)
    assert group_layout == [(g.tag, dict(g.attrib), g.getparent()) for g in node.iter(*GROUP_TAGS)]
    assert set(before_ids) - set(after_ids) == deleted and removed.isdisjoint(deleted)
    removed.update(deleted)
    assert after_ids
    singleton = len(after_ids) == 1
    if singleton:
        gid = after_ids[0]
        protein, taxon = training_genes[gid]
        singleton_rows.append(dict(root_hog=hog, orthoxml_id=exact, gene_id=gid,
                                   protein=protein, taxon=taxon, reason="singleton_root_hog"))
        discard.add(gid)
    else:
        edited_hashes[exact] = tree_hash(node)
        retained_training.update(gid for gid in after_ids if gid in training_genes)
    rows.append(dict(root_hog=hog, excluded_singleton=singleton, oma_leaves_removed=len(deleted),
                     **{f"before_{key}": value for key, value in before.items()}, **after))
assert len(rows) == len(train) and len(removed) == sum(row["oma_leaves_removed"] for row in rows)
assert len(removed) == 67731 and len(singleton_rows) == 417
assert removed.isdisjoint(discard)
metrics = pd.DataFrame(rows)
metrics.to_csv(OUT / "tree_metrics.tsv", sep="\t", index=False)
pd.DataFrame(singleton_rows).to_csv(OUT / "uncovered_singleton_proteins.tsv", sep="\t", index=False)
assigned_training = retained_training | discard
assert len(assigned_training) == train.CAFA_train_leaves.sum()
assert retained_training.isdisjoint(discard)
unassigned = training_genes.keys() - assigned_training
pd.DataFrame([dict(gene_id=gid, protein=training_genes[gid][0], taxon=training_genes[gid][1],
                   reason="not_assigned_to_root_hog")
              for gid in sorted(unassigned, key=int)]).to_csv(
    OUT / "uncovered_unassigned_proteins.tsv", sep="\t", index=False
)
coverage = dict(input_cafa_train=len(training_genes), assigned_cafa_train=len(assigned_training),
                retained_cafa_train=len(retained_training), singleton_cafa_train=len(discard),
                unassigned_cafa_train=len(unassigned), uncovered_cafa_train=len(discard | unassigned))
display(pd.Series(coverage).to_frame("CAFA-training proteins"))
print(f"Verified {len(removed):,} OMA removals and {len(singleton_rows):,} singleton exclusions.", flush=True)


## 9. Compare tree structure before and after filtering

Show root and leaf counts at each stage and reproduce the five structure plots for the retained HOGs. Unary groups and branching speciation-like events are separate counts. Species counts merge mapped NCBI IDs and use distinct OMA genome codes where a taxon mapping is unavailable.

In [ ]:
retained_metrics = metrics.loc[~metrics.excluded_singleton]
display(pd.DataFrame({
    "Unpruned subset": [len(metrics), metrics.before_leaves.sum(), metrics.before_unary_nodes.sum()],
    "After OMA removal": [len(metrics), metrics.leaves.sum(), metrics.unary_nodes.sum()],
    "After singleton exclusion": [len(retained_metrics), retained_metrics.leaves.sum(), retained_metrics.unary_nodes.sum()],
}, index=["Root HOGs", "Protein leaves", "Unary nodes"]))
plot_pairs = [
    ("duplications", "multifurcations", "Duplications", "Multifurcations"),
    ("leaves", "multifurcations", "Protein leaves", "Multifurcations"),
    ("leaves", "duplications", "Protein leaves", "Duplications"),
    ("species_units", "duplications", "Species units", "Duplications"),
    ("species_units", "speciations", "Species units", "Branching speciation-like nodes"),
]
for index, (x, y, xlabel, ylabel) in enumerate(plot_pairs, start=7):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(retained_metrics[x], retained_metrics[y], s=7, alpha=.17, color="#28667f", linewidths=0)
    ax.set_xscale("symlog", linthresh=1)
    ax.set_yscale("symlog", linthresh=1)
    ax.set(xlabel=xlabel, ylabel=ylabel, title=f"{ylabel} vs {xlabel.lower()}")
    fig.tight_layout()
    path = OUT / f"{index:02d}_{y}_vs_{x}.png"
    fig.savefig(path, dpi=145, bbox_inches="tight")
    plt.close(fig)
    display(Image(filename=str(path)))

## 10. Write and validate the final HOGprop OrthoXML

Write the retained original root groups with only the approved leaf removals. Filter the registry to the remaining references. Compare every written group against its validated post-removal fingerprint, reconcile counts and test loading all HOGs with HOGprop before publishing the file. The original unary groups may violate the strict OrthoXML minimum child count. The compatibility check uses the local HOGprop loader.

In [ ]:
class FinalFilter(StreamingProfessorFilter):
    def process_species(self, element):
        for gene in element.iter(GENE):
            if gene.get("id") in removed or gene.get("id") in discard:
                gene.getparent().remove(gene)
        return element

    def process_toplevel_group(self, element):
        exact = element.get("id")
        if exact not in edited_hashes:
            return None
        apply_removals(element, exact.split("_", 1)[0], verify=False)
        return element

pending = FINAL.with_suffix(".pending.orthoxml")
professor.process_stream_orthoxml(XML, str(pending), parser_cls=FinalFilter)
retained = metrics.loc[~metrics.excluded_singleton]
leaf_count = audit_xml(pending, edited_hashes, removed | discard)
assert leaf_count == retained.leaves.sum()

# The loader check is defined here so the whole workflow lives in the notebook.
hogprop_code = """
import json
import sys
from collections import Counter
import numpy as np
sys.path.insert(0, sys.argv[1])
from hogprop import __version__
from hogprop.HOGParser import HOGParser, HOGLevelIDs
from hogprop.HOGArray import HOGArray
counts = Counter()
with HOGParser(sys.argv[2], cache_species=False) as parser:
    counts["species"] = len(list(parser.species))
    for hog in parser.HOGs(auto_clean=True):
        assert HOGLevelIDs(parser).get_map(hog)
        types, _, ranges, _ = HOGArray(hog).arrays
        degrees = ranges[:, 1] - ranges[:, 0]
        leaves = int(np.count_nonzero(types == 0))
        assert leaves > 1
        counts["roots"] += 1
        counts["leaves"] += leaves
        counts["unary_nodes"] += int(np.count_nonzero(degrees == 1))
        counts["branching_speciations"] += int(np.count_nonzero((types == 1) & (degrees >= 2)))
        hog.get_distance.cache_clear()
print(json.dumps({"version": __version__, "counts": counts}))
"""
hogprop_python = os.environ.get("HOGPROP_PYTHON", sys.executable)
hogprop_checkout = REPO / "software/hogprop"
loader_output = subprocess.check_output(
    [hogprop_python, "-c", hogprop_code, str(hogprop_checkout), str(pending)], text=True
)
loaded = json.loads(loader_output)
assert loaded["counts"]["roots"] == len(retained)
assert loaded["counts"]["leaves"] == leaf_count
assert loaded["counts"]["unary_nodes"] == retained.unary_nodes.sum()
assert loaded["counts"]["branching_speciations"] == retained.speciations.sum()
pending.replace(FINAL)

from importlib.metadata import version
evidence = {
    "source_sha256": source_sha,
    "professor_script_sha256": file_hash(PROFESSOR),
    "assignments_sha256": file_hash(ASSIGNMENTS),
    "mapping_sha256": file_hash(MAP),
    "unpruned_sha256": xml_hash.hexdigest(),
    "final_sha256": file_hash(FINAL),
    "python": sys.version,
    "orthoxml_tools": version("orthoxml-tools"),
    "lxml": ".".join(map(str, etree.LXML_VERSION)),
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
    "coverage": coverage,
    "source_roots_selected": len(train),
    "oma_leaves_removed": len(removed),
    "singleton_roots_excluded": len(singleton_rows),
    "retained_roots": len(retained),
    "retained_leaves": leaf_count,
    "unary_nodes_retained": int(retained.unary_nodes.sum()),
    "branching_speciation_nodes": int(retained.speciations.sum()),
    "internal_nodes_collapsed": 0,
    "hogprop": loaded,
}
(OUT / "validation.json").write_text(json.dumps(evidence, indent=2) + "\n")
display(pd.Series({key: value for key, value in evidence.items()
                   if type(value) is int}).to_frame("Verified count"))
print(FINAL)
